In [1]:
# Importar la librería para manipulación de datos tabulares
import pandas as pd
# Importar la librería para operaciones numéricas y arreglos
import numpy as np
# Importar el modelo de regresión lineal para la ortogonalización
from sklearn.linear_model import LinearRegression
# Importar el escalador estándar para normalizar variables antes del clustering
from sklearn.preprocessing import StandardScaler
# Importar los algoritmos estadísticos de clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
# Importar las métricas matemáticas de evaluación de clústeres
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ==============================================================================
# FASE 1: PREPROCESAMIENTO Y ORTOGONALIZACIÓN (ELIMINACIÓN DE SESGO DE AGUA)
# ==============================================================================

# 1. Cargar el dataset original en crudo desde el archivo CSV
df_crudo = pd.read_csv('../data/raw/intensidades.csv')

# 2. Definir explícitamente la lista de variables objetivo (canales de metales)
metales = ['n1fe', 'n2cu', 'n3zn', 'n4mo']

# 3. Calcular el umbral del percentil 90 para el canal de dispersión (agua)
umbral_agua_p90 = df_crudo['n6sc'].quantile(0.90)

# 4. Filtrar el dataframe reteniendo solo los estados estacionarios (sin flushing)
# Se usa .copy() para crear un dataframe independiente en memoria
df_filtrado = df_crudo[df_crudo['n6sc'] <= umbral_agua_p90].copy()

# 5. Iniciar un bucle para corregir cada metal iterativamente
for metal in metales:
    # Definir la variable independiente (X): El canal de dispersión (agua)
    X_agua = df_filtrado[['n6sc']]
    
    # Definir la variable dependiente (y): La intensidad cruda del metal actual
    y_metal_crudo = df_filtrado[metal]
    
    # Instanciar el modelo de regresión lineal simple
    modelo_regresion = LinearRegression()
    
    # Entrenar el modelo para encontrar la relación entre el agua y la caída del metal
    modelo_regresion.fit(X_agua, y_metal_crudo)
    
    # Predecir cuánta intensidad se pierde puramente por efecto de la dilución
    metal_esperado_por_dilucion = modelo_regresion.predict(X_agua)
    
    # Calcular el valor medio de la intensidad del metal para restaurar la escala física
    media_metal = y_metal_crudo.mean()
    
    # Calcular el residuo (valor real - valor espurio) y sumar la media
    metal_corregido_flotante = (y_metal_crudo - metal_esperado_por_dilucion) + media_metal
    
    # Redondear el valor al entero más cercano respetando la física de conteos de fotones
    metal_corregido_redondeado = metal_corregido_flotante.round(0)
    
    # Convertir el tipo de dato a entero explícitamente y guardarlo en una nueva columna
    df_filtrado[f'{metal}_corregido'] = metal_corregido_redondeado.astype(int)

# 6. Crear una lista explícita de las columnas base que queremos conservar
columnas_base = ['date', 'time', 'instance']

# 7. Crear una lista explícita con los nombres de las nuevas columnas corregidas
columnas_corregidas = [f'{metal}_corregido' for metal in metales]

# 8. Concatenar ambas listas para definir las columnas finales a mantener
columnas_finales = columnas_base + columnas_corregidas

# 9. Seleccionar solo las columnas de interés en el dataframe filtrado
df_limpio = df_filtrado[columnas_finales]

# 10. Ordenar cronológicamente el dataframe usando fecha y hora
df_limpio = df_limpio.sort_values(['date', 'time'])

# 11. Reiniciar el índice del dataframe para que sea continuo desde 0
df_limpio = df_limpio.reset_index(drop=True)

# ==============================================================================
# FASE 2: EVALUACIÓN DE CLUSTERING GEOMETALÚRGICO
# ==============================================================================

# 12. Extraer los valores numéricos puros (matriz X) para el clustering
matriz_X = df_limpio[columnas_corregidas].values

# 13. Instanciar el escalador estándar (Z-score normalization)
escalador = StandardScaler()

# 14. Ajustar el escalador y transformar la matriz X a media 0 y varianza 1
matriz_X_escalada = escalador.fit_transform(matriz_X)

# 15. Crear una lista vacía para almacenar los resultados de las métricas
resultados_metricas = []

# 16. Definir una función explícita para evaluar los clústeres
def evaluar_y_guardar_modelo(nombre_algoritmo, etiquetas_predichas, datos_escalados):
    # Encontrar las etiquetas únicas generadas por el modelo
    etiquetas_unicas = np.unique(etiquetas_predichas)
    
    # Crear una máscara booleana para excluir el ruido detectado (etiqueta -1 en DBSCAN)
    mascara_validos = etiquetas_predichas != -1
    
    # Filtrar las etiquetas reteniendo solo las asignaciones válidas
    etiquetas_validas = etiquetas_predichas[mascara_validos]
    
    # Contar la cantidad de clústeres válidos formados (debe ser mayor a 1 para evaluar)
    cantidad_clusteres = len(set(etiquetas_validas))
    
    # Condición lógica: solo evaluar si hay al menos 2 clústeres reales
    if cantidad_clusteres > 1:
        # Filtrar la matriz de datos excluyendo el ruido
        datos_validos = datos_escalados[mascara_validos]
        
        # Calcular el coeficiente de Silhouette (cohesión y separación)
        score_silueta = silhouette_score(datos_validos, etiquetas_validas)
        
        # Calcular el índice Davies-Bouldin (dispersión interna vs separación externa)
        score_db = davies_bouldin_score(datos_validos, etiquetas_validas)
        
        # Calcular el índice Calinski-Harabasz (ratio de varianza entre y dentro de clusters)
        score_ch = calinski_harabasz_score(datos_validos, etiquetas_validas)
        
        # Guardar las métricas calculadas como un diccionario en la lista de resultados
        resultados_metricas.append({
            'Algoritmo': nombre_algoritmo,
            'Clusters_Reales': cantidad_clusteres,
            'Silhouette_Score': round(score_silueta, 4),
            'Davies_Bouldin': round(score_db, 4),
            'Calinski_Harabasz': round(score_ch, 1)
        })

# 17. Evaluar el algoritmo K-Means iterando desde 2 hasta 6 clústeres
for k_clusters in range(2, 7):
    # Instanciar K-Means fijando la semilla aleatoria para reproducibilidad
    modelo_kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    # Entrenar el modelo y obtener las etiquetas asignadas a cada punto
    etiquetas_kmeans = modelo_kmeans.fit_predict(matriz_X_escalada)
    # Llamar a la función de evaluación pasando el nombre, etiquetas y datos
    evaluar_y_guardar_modelo(f'K-Means_k{k_clusters}', etiquetas_kmeans, matriz_X_escalada)

# 18. Evaluar el algoritmo Gaussian Mixture (GMM) iterando desde 2 hasta 6 clústeres
for gmm_clusters in range(2, 7):
    # Instanciar GMM fijando la semilla aleatoria
    modelo_gmm = GaussianMixture(n_components=gmm_clusters, random_state=42)
    # Entrenar GMM y obtener etiquetas
    etiquetas_gmm = modelo_gmm.fit_predict(matriz_X_escalada)
    # Llamar a la función de evaluación
    evaluar_y_guardar_modelo(f'GMM_k{gmm_clusters}', etiquetas_gmm, matriz_X_escalada)

# 19. Evaluar Clustering Aglomerativo Jerárquico iterando desde 2 hasta 6 clústeres
for aglom_clusters in range(2, 7):
    # Instanciar el modelo Aglomerativo (no requiere semilla aleatoria en su formulación base)
    modelo_aglomerativo = AgglomerativeClustering(n_clusters=aglom_clusters)
    # Entrenar y obtener etiquetas
    etiquetas_aglom = modelo_aglomerativo.fit_predict(matriz_X_escalada)
    # Llamar a la función de evaluación
    evaluar_y_guardar_modelo(f'Aglomerativo_k{aglom_clusters}', etiquetas_aglom, matriz_X_escalada)

# 20. Evaluar DBSCAN iterando sobre una grilla de hiperparámetros
# Definir lista de valores para epsilon (radio de vecindad)
lista_eps = [0.5, 0.8, 1.0, 1.2]
# Definir lista de valores para muestras mínimas (densidad requerida)
lista_min_samples = [4, 5, 10]

# Bucle anidado para explorar combinaciones de hiperparámetros de DBSCAN
for eps_actual in lista_eps:
    for min_samp_actual in lista_min_samples:
        # Instanciar DBSCAN con los parámetros actuales
        modelo_dbscan = DBSCAN(eps=eps_actual, min_samples=min_samp_actual)
        # Entrenar DBSCAN y obtener etiquetas (incluirá ruido como -1)
        etiquetas_dbscan = modelo_dbscan.fit_predict(matriz_X_escalada)
        # Construir el nombre del modelo con sus parámetros explícitos
        nombre_dbscan = f'DBSCAN_eps{eps_actual}_ms{min_samp_actual}'
        # Llamar a la función de evaluación
        evaluar_y_guardar_modelo(nombre_dbscan, etiquetas_dbscan, matriz_X_escalada)

# ==============================================================================
# FASE 3: SELECCIÓN DEL MEJOR MODELO Y EXPORTACIÓN FINAL
# ==============================================================================

# 21. Convertir la lista de resultados de métricas en un Dataframe de Pandas
df_resultados = pd.DataFrame(resultados_metricas)

# 22. Ordenar los resultados descendientemente según el score de Silhouette (el principal criterio)
df_resultados_ordenados = df_resultados.sort_values(by='Silhouette_Score', ascending=False)

# 23. Reiniciar el índice para acceder limpiamente a la primera fila (el mejor)
df_resultados_ordenados = df_resultados_ordenados.reset_index(drop=True)

# 24. Imprimir la tabla completa de evaluaciones para revisión técnica
print("=== TABLA DE EVALUACIÓN DE CLUSTERING ===")
print(df_resultados_ordenados.to_string())

# 25. Extraer dinámicamente el nombre y número de clusters del mejor modelo en la fila 0
mejor_algoritmo_nombre = df_resultados_ordenados.loc[0, 'Algoritmo']
mejor_cantidad_clusters = int(df_resultados_ordenados.loc[0, 'Clusters_Reales'])

# 26. Imprimir el ganador
print(f"\n[INFO] El mejor modelo según Silhouette es: {mejor_algoritmo_nombre}")

# 27. Lógica condicional para instanciar, re-entrenar el modelo ganador y asignar la etiqueta
if 'K-Means' in mejor_algoritmo_nombre:
    mejor_modelo = KMeans(n_clusters=mejor_cantidad_clusters, random_state=42, n_init=10)
    df_limpio['Cluster_Asignado'] = mejor_modelo.fit_predict(matriz_X_escalada)

elif 'GMM' in mejor_algoritmo_nombre:
    mejor_modelo = GaussianMixture(n_components=mejor_cantidad_clusters, random_state=42)
    df_limpio['Cluster_Asignado'] = mejor_modelo.fit_predict(matriz_X_escalada)

elif 'Aglomerativo' in mejor_algoritmo_nombre:
    mejor_modelo = AgglomerativeClustering(n_clusters=mejor_cantidad_clusters)
    df_limpio['Cluster_Asignado'] = mejor_modelo.fit_predict(matriz_X_escalada)

elif 'DBSCAN' in mejor_algoritmo_nombre:
    # Extraer los parámetros de la cadena (Ej. DBSCAN_eps0.5_ms10)
    # Dividiendo el string por guiones bajos
    partes_nombre = mejor_algoritmo_nombre.split('_')
    # Extraer epsilon quitando el prefijo 'eps' y convirtiendo a float
    mejor_eps = float(partes_nombre[1].replace('eps', ''))
    # Extraer min_samples quitando el prefijo 'ms' y convirtiendo a int
    mejor_ms = int(partes_nombre[2].replace('ms', ''))
    
    # Instanciar DBSCAN con los hiperparámetros ganadores
    mejor_modelo = DBSCAN(eps=mejor_eps, min_samples=mejor_ms)
    df_limpio['Cluster_Asignado'] = mejor_modelo.fit_predict(matriz_X_escalada)

# 28. Guardar el dataframe final (limpio, ortogonalizado y clasificado) en un CSV para el siguiente paso
df_limpio.to_csv('pipeline_metalurgico_final.csv', index=False)

print("\n[INFO] Ejecución completada. Archivo 'pipeline_metalurgico_final.csv' generado exitosamente.")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/intensidades.csv'